In [1]:
import numpy as np
import pandas as pd

from sklearn.datasets import load_breast_cancer

data=load_breast_cancer()
x=data.data
y=data.target


In [6]:
print("Feature shape:", x.shape)
print("Target shape:", y.shape)
print("\nTarget labels:")
print(data.target_names)


Feature shape: (569, 30)
Target shape: (569,)

Target labels:
['malignant' 'benign']


In [7]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score,confusion_matrix,roc_auc_score

x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)
x_train.shape

(455, 30)

In [8]:
model=RandomForestClassifier(n_estimators=50,max_depth=2,min_samples_leaf=3,min_samples_split=3,max_features='log2')
model.fit(x_train,y_train)
y_pred=model.predict(x_test)
print('accuracy_score:',accuracy_score(y_test,y_pred))
print('accuracy_score:',roc_auc_score(y_test,y_pred))

accuracy_score: 0.9649122807017544
accuracy_score: 0.9580740255486406


In [9]:
import optuna

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

def objective(trial):

    params={
        'n_estimators':trial.suggest_int('n_estimators',50,500,step=50),
        'max_depth':trial.suggest_int('max_depth',2,20),
        'min_samples_split':trial.suggest_int('min_samples_split',2,20),
        'min_samples_leaf':trial.suggest_int('min_samples_leaf',1,10),
        'max_features':trial.suggest_categorical('max_features',['sqrt','log2',None]),
        'bootstrap': trial.suggest_categorical("bootstrap", [True, False]),
        "random_state": 42,
        "n_jobs": -1
    }

    model=RandomForestClassifier(**params)
    scores=cross_val_score(model,x_train,y_train,cv=5,scoring='accuracy',n_jobs=-1)

    return scores.mean()

In [10]:
study=optuna.create_study(direction='maximize')
study.optimize(objective,n_trials=50)
print("Best CV accuracy:", study.best_value)
print("Best hyperparameters:", study.best_params)

[I 2026-10-24 23:07:26,745] A new study created in memory with name: no-name-1aebcd8d-4b50-4138-b37a-f19f184c00f5
[I 2026-10-24 23:07:36,523] Trial 0 finished with value: 0.9252747252747252 and parameters: {'n_estimators': 150, 'max_depth': 19, 'min_samples_split': 14, 'min_samples_leaf': 6, 'max_features': None, 'bootstrap': False}. Best is trial 0 with value: 0.9252747252747252.
[I 2026-10-24 23:07:42,116] Trial 1 finished with value: 0.9252747252747252 and parameters: {'n_estimators': 150, 'max_depth': 15, 'min_samples_split': 20, 'min_samples_leaf': 5, 'max_features': None, 'bootstrap': False}. Best is trial 0 with value: 0.9252747252747252.
[I 2026-10-24 23:07:43,512] Trial 2 finished with value: 0.9494505494505494 and parameters: {'n_estimators': 250, 'max_depth': 4, 'min_samples_split': 7, 'min_samples_leaf': 8, 'max_features': 'log2', 'bootstrap': True}. Best is trial 2 with value: 0.9494505494505494.
[I 2026-10-24 23:07:43,934] Trial 3 finished with value: 0.945054945054945 an

Best CV accuracy: 0.9582417582417582
Best hyperparameters: {'n_estimators': 500, 'max_depth': 7, 'min_samples_split': 20, 'min_samples_leaf': 6, 'max_features': 'sqrt', 'bootstrap': False}
